In [ ]:
%pip uninstall -y tensorflow keras tf-keras tensorflow-text tensorflow-hub tf-models-official >/dev/null 2>&1

%pip install -q \
  "tensorflow==2.19.0" \
  "tensorflow-text==2.19.0" \
  "tensorflow-hub==0.16.1" \
  "tf-keras==2.19.0" \
  "pandas" \
  "scikit-learn" \
  "numpy"

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
import math
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import tf_keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

print("TF:", tf.__version__)
print("Hub:", hub.__version__)
print("TF Text:", text.__version__)
print("tf_keras:", tf_keras.__version__)
print("tf.keras module:", tf.keras)

TF: 2.19.0
Hub: 0.16.1
TF Text: 2.19.0
tf_keras: 2.19.0
tf.keras module: <module 'tf_keras.api._v2.keras' from '/usr/local/lib/python3.12/dist-packages/tf_keras/api/_v2/keras/__init__.py'>


In [ ]:

from dataclasses import dataclass, asdict
from typing import Tuple
import re

# =========================
# Configuration
# =========================
CSV_PATH = "civic_priority_dataset_6000.csv"  # <- change this if needed
OUTPUT_DIR = "priority_mobile_outputs"

TITLE_COL = "Title"
DESCRIPTION_COL = "Description"
CATEGORY_COL = "Category"
LABEL_COL = "PriorityScore"

# Strong teacher for quality. It is used only during training/distillation.
PREPROCESS_HANDLE = "https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3"
ENCODER_HANDLE = "https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-512_A-8/1"
TRAIN_TEACHER = True

# Split / training
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15
RANDOM_STATE = 42

TEACHER_MAX_EPOCHS = 5
TEACHER_BATCH_SIZE = 16
TEACHER_LEARNING_RATE = 3e-5

STUDENT_MAX_EPOCHS = 20
STUDENT_BATCH_SIZE = 64
STUDENT_LEARNING_RATE = 2e-3

# Student model: fully Android-friendly numeric-input model.
VOCAB_SIZE = 20000
MAX_SEQUENCE_LENGTH = 96
EMBED_DIM = 128
CONV_FILTERS = 128
DROPOUT = 0.25
DISTILL_BLEND = 0.35  # 35% teacher guidance, 65% true labels

ENABLE_DYNAMIC_RANGE_QUANT = True

TEXT_TEMPLATE = (
    "title: {title} "
    "category: {category} "
    "description: {description} "
    "important_title: {title}"
)


tf.keras.utils.set_random_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


@dataclass
class ExportConfig:
    output_dir: str
    title_col: str
    description_col: str
    category_col: str
    label_col: str
    preprocess_handle: str
    encoder_handle: str
    train_teacher: bool
    teacher_max_epochs: int
    teacher_batch_size: int
    teacher_learning_rate: float
    student_max_epochs: int
    student_batch_size: int
    student_learning_rate: float
    validation_size: float
    test_size: float
    random_state: int
    vocab_size: int
    max_sequence_length: int
    embed_dim: int
    conv_filters: int
    dropout: float
    distill_blend: float
    text_template: str


# =========================
# Text helpers
# =========================
def build_structured_text(df: pd.DataFrame) -> pd.Series:
    title = df[TITLE_COL].fillna("").astype(str).str.strip()
    description = df[DESCRIPTION_COL].fillna("").astype(str).str.strip()
    category = df[CATEGORY_COL].fillna("").astype(str).str.strip()

    return (
        "title: " + title +
        " category: " + category +
        " description: " + description +
        " important_title: " + title
    )


def normalize_text_py(value: str) -> str:
    value = "" if value is None else str(value)
    value = value.lower()
    value = re.sub(r"[^a-z0-9:_ ]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def tf_standardize_text(x: tf.Tensor) -> tf.Tensor:
    x = tf.strings.lower(x)
    x = tf.strings.regex_replace(x, r"[^a-z0-9:_ ]+", " ")
    x = tf.strings.regex_replace(x, r"\s+", " ")
    return tf.strings.strip(x)


# =========================
# Data loading / splitting
# =========================
def load_dataframe(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    required = {TITLE_COL, DESCRIPTION_COL, CATEGORY_COL, LABEL_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df = df.copy()
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
    df = df.dropna(subset=[LABEL_COL])
    df[LABEL_COL] = df[LABEL_COL].clip(0.0, 1.0).astype("float32")
    df["model_text"] = build_structured_text(df)
    df = df[df["model_text"].str.len() > 0].reset_index(drop=True)

    if len(df) < 100:
        print("WARNING: dataset is very small. A mobile model may overfit badly.")

    return df


def make_stratify_labels(values: pd.Series):
    try:
        bins = pd.qcut(values, q=5, labels=False, duplicates="drop")
        counts = pd.Series(bins).value_counts()
        if len(counts) < 2 or counts.min() < 2:
            return None
        return bins
    except Exception:
        return None


def split_dataframe(df: pd.DataFrame):
    stratify_1 = make_stratify_labels(df[LABEL_COL])
    try:
        train_df, temp_df = train_test_split(
            df,
            test_size=VALIDATION_SIZE + TEST_SIZE,
            random_state=RANDOM_STATE,
            shuffle=True,
            stratify=stratify_1 if stratify_1 is not None else None,
        )
    except ValueError:
        train_df, temp_df = train_test_split(
            df,
            test_size=VALIDATION_SIZE + TEST_SIZE,
            random_state=RANDOM_STATE,
            shuffle=True,
        )

    relative_test_size = TEST_SIZE / (VALIDATION_SIZE + TEST_SIZE)
    stratify_2 = make_stratify_labels(temp_df[LABEL_COL])
    try:
        val_df, test_df = train_test_split(
            temp_df,
            test_size=relative_test_size,
            random_state=RANDOM_STATE,
            shuffle=True,
            stratify=stratify_2 if stratify_2 is not None else None,
        )
    except ValueError:
        val_df, test_df = train_test_split(
            temp_df,
            test_size=relative_test_size,
            random_state=RANDOM_STATE,
            shuffle=True,
        )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


# =========================
# Metrics
# =========================
def score_band(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    return np.digitize(values, bins=[0.33, 0.66], right=False)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    mae = float(np.mean(np.abs(y_true - y_pred)))
    mse = float(np.mean((y_true - y_pred) ** 2))
    rmse = float(math.sqrt(mse))

    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - (ss_res / ss_tot)) if ss_tot > 0 else float("nan")
    band_acc = float(np.mean(score_band(y_true) == score_band(y_pred)))

    return {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
        "band_accuracy": band_acc,
    }


def to_serializable(value):
    if isinstance(value, dict):
        return {str(k): to_serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_serializable(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, pd.Series):
        return [to_serializable(v) for v in value.tolist()]
    if isinstance(value, pd.Index):
        return [to_serializable(v) for v in value.tolist()]
    return value


def write_json(path: str, payload) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_serializable(payload), f, indent=2)


# =========================
# Teacher (quality)
# =========================
def make_teacher_dataset(frame: pd.DataFrame, training: bool) -> tf.data.Dataset:
    x = frame["model_text"].astype(str).tolist()
    y = frame[LABEL_COL].astype("float32").values
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(min(len(frame), 4096), seed=RANDOM_STATE, reshuffle_each_iteration=True)
    ds = ds.batch(TEACHER_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


def build_teacher_model() -> tf.keras.Model:
    text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name="text")

    preprocess = hub.KerasLayer(PREPROCESS_HANDLE, name="preprocessing")
    encoder_inputs = preprocess(text_input)

    encoder = hub.KerasLayer(
        ENCODER_HANDLE,
        trainable=True,
        name="bert_encoder",
    )
    encoder_outputs = encoder(encoder_inputs)
    pooled_output = encoder_outputs["pooled_output"]

    x = tf.keras.layers.Dropout(DROPOUT, name="dropout")(pooled_output)
    x = tf.keras.layers.Dense(128, activation="relu", name="dense_128")(x)
    x = tf.keras.layers.Dropout(DROPOUT, name="dropout_2")(x)
    score = tf.keras.layers.Dense(1, activation="sigmoid", name="priority_score")(x)

    return tf.keras.Model(inputs=text_input, outputs=score, name="priority_teacher")


def compile_teacher_model(model: tf.keras.Model) -> tf.keras.Model:
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=TEACHER_LEARNING_RATE,
        weight_decay=0.01,
    )
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.Huber(),
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.MeanSquaredError(name="mse"),
        ],
    )
    return model


# =========================
# Student preprocessing
# =========================
def build_vectorizer(train_texts) -> tf.keras.layers.TextVectorization:
    vectorizer = tf.keras.layers.TextVectorization(
        max_tokens=VOCAB_SIZE,
        standardize=tf_standardize_text,
        split="whitespace",
        output_mode="int",
        output_sequence_length=MAX_SEQUENCE_LENGTH,
        name="priority_vectorizer",
    )
    adapt_ds = tf.data.Dataset.from_tensor_slices(list(train_texts)).batch(256)
    vectorizer.adapt(adapt_ds)
    return vectorizer


def vectorize_texts(vectorizer, texts, batch_size: int = 2048) -> np.ndarray:
    texts = list(texts)
    chunks = []
    for start in range(0, len(texts), batch_size):
        batch = tf.constant(texts[start:start + batch_size])
        chunks.append(vectorizer(batch).numpy().astype("int32"))
    return np.concatenate(chunks, axis=0)


def make_vocab_lookup(vocab):
    return {token: idx for idx, token in enumerate(vocab)}


def encode_texts_with_vocab(texts, vocab_lookup) -> np.ndarray:
    encoded = np.zeros((len(texts), MAX_SEQUENCE_LENGTH), dtype=np.int32)
    for i, text_value in enumerate(texts):
        tokens = normalize_text_py(text_value).split()
        token_ids = [vocab_lookup.get(token, 1) for token in tokens[:MAX_SEQUENCE_LENGTH]]
        encoded[i, :len(token_ids)] = token_ids
    return encoded


def verify_manual_tokenizer(vectorizer, sample_texts) -> None:
    vocab = vectorizer.get_vocabulary()
    vocab_lookup = make_vocab_lookup(vocab)

    tf_ids = vectorize_texts(vectorizer, sample_texts)
    manual_ids = encode_texts_with_vocab(sample_texts, vocab_lookup)

    if not np.array_equal(tf_ids, manual_ids):
        raise AssertionError(
            "Manual tokenizer does not match TextVectorization output. "
            "Do not export until this passes."
        )


# =========================
# Student (mobile)
# =========================
def make_student_dataset(token_ids: np.ndarray, labels: np.ndarray, training: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((token_ids.astype("int32"), labels.astype("float32")))
    if training:
        ds = ds.shuffle(min(len(token_ids), 8192), seed=RANDOM_STATE, reshuffle_each_iteration=True)
    ds = ds.batch(STUDENT_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


def build_student_model(actual_vocab_size: int) -> tf.keras.Model:
    token_input = tf.keras.layers.Input(
        shape=(MAX_SEQUENCE_LENGTH,),
        dtype=tf.int32,
        name="token_ids",
    )

    x = tf.keras.layers.Embedding(
        input_dim=actual_vocab_size,
        output_dim=EMBED_DIM,
        name="token_embedding",
    )(token_input)
    x = tf.keras.layers.SpatialDropout1D(DROPOUT, name="embedding_dropout")(x)

    pooled_features = []
    for kernel_size in (3, 5, 7):
        branch = tf.keras.layers.Conv1D(
            filters=CONV_FILTERS,
            kernel_size=kernel_size,
            padding="same",
            activation="relu",
            name=f"conv_{kernel_size}",
        )(x)
        pooled_features.append(
            tf.keras.layers.GlobalMaxPooling1D(name=f"global_max_{kernel_size}")(branch)
        )
        pooled_features.append(
            tf.keras.layers.GlobalAveragePooling1D(name=f"global_avg_{kernel_size}")(branch)
        )

    x = tf.keras.layers.Concatenate(name="pooled_features")(pooled_features)
    x = tf.keras.layers.Dense(256, activation="relu", name="dense_256")(x)
    x = tf.keras.layers.Dropout(DROPOUT, name="dropout_1")(x)
    x = tf.keras.layers.Dense(64, activation="relu", name="dense_64")(x)
    x = tf.keras.layers.Dropout(DROPOUT, name="dropout_2")(x)
    score = tf.keras.layers.Dense(1, activation="sigmoid", name="priority_score")(x)

    return tf.keras.Model(inputs=token_input, outputs=score, name="priority_mobile_student")


def compile_student_model(model: tf.keras.Model) -> tf.keras.Model:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=STUDENT_LEARNING_RATE,
            clipnorm=1.0,
        ),
        loss=tf.keras.losses.Huber(),
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.MeanSquaredError(name="mse"),
        ],
    )
    return model


# =========================
# Export helpers
# =========================
def save_config(out_dir: str, actual_vocab_size: int, teacher_available: bool):
    cfg = asdict(ExportConfig(
        output_dir=OUTPUT_DIR,
        title_col=TITLE_COL,
        description_col=DESCRIPTION_COL,
        category_col=CATEGORY_COL,
        label_col=LABEL_COL,
        preprocess_handle=PREPROCESS_HANDLE,
        encoder_handle=ENCODER_HANDLE,
        train_teacher=TRAIN_TEACHER,
        teacher_max_epochs=TEACHER_MAX_EPOCHS,
        teacher_batch_size=TEACHER_BATCH_SIZE,
        teacher_learning_rate=TEACHER_LEARNING_RATE,
        student_max_epochs=STUDENT_MAX_EPOCHS,
        student_batch_size=STUDENT_BATCH_SIZE,
        student_learning_rate=STUDENT_LEARNING_RATE,
        validation_size=VALIDATION_SIZE,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        vocab_size=VOCAB_SIZE,
        max_sequence_length=MAX_SEQUENCE_LENGTH,
        embed_dim=EMBED_DIM,
        conv_filters=CONV_FILTERS,
        dropout=DROPOUT,
        distill_blend=DISTILL_BLEND,
        text_template=TEXT_TEMPLATE,
    ))
    cfg.update({
        "actual_vocab_size": int(actual_vocab_size),
        "teacher_available": bool(teacher_available),
        "pad_token_id": 0,
        "oov_token_id": 1,
        "input_name": "token_ids",
        "input_dtype": "int32",
        "input_shape": [1, MAX_SEQUENCE_LENGTH],
        "output_name": "priority_score",
        "output_range": [0.0, 1.0],
        "tflite_model": "priority_mobile_regressor.tflite",
        "vocab_file": "vocabulary.txt",
        "normalization": {
            "lowercase": True,
            "regex_replace": r"[^a-z0-9:_ ]+",
            "collapse_whitespace": True,
            "trim": True,
        },
    })

    write_json(os.path.join(out_dir, "training_config.json"), cfg)


def save_vocabulary(vectorizer, out_dir: str):
    vocab = vectorizer.get_vocabulary()
    with open(os.path.join(out_dir, "vocabulary.txt"), "w", encoding="utf-8") as f:
        for token in vocab:
            f.write(token + "\n")
    return vocab


def save_android_reference(out_dir: str):
    kotlin_lines = [
        "import java.util.Locale",
        "",
        "object PriorityTokenizer {",
        f"    const val MAX_SEQUENCE_LENGTH = {MAX_SEQUENCE_LENGTH}",
        "    private val invalidChars = Regex(\"[^a-z0-9:_ ]+\")",
        "    private val multiSpace = Regex(\"\\s+\")",
        "",
        "    fun buildModelText(title: String, description: String, category: String): String {",
        "        return \"title: $title category: $category description: $description important_title: $title\"",
        "    }",
        "",
        "    fun normalize(raw: String): String {",
        "        return raw",
        "            .lowercase(Locale.US)",
        "            .replace(invalidChars, \" \")",
        "            .replace(multiSpace, \" \")",
        "            .trim()",
        "    }",
        "",
        "    fun encode(title: String, description: String, category: String, vocab: Map<String, Int>): IntArray {",
        "        val normalized = normalize(buildModelText(title, description, category))",
        "        val tokens = normalized.split(\" \").filter { it.isNotBlank() }",
        "        val ids = IntArray(MAX_SEQUENCE_LENGTH)",
        "        val limit = minOf(tokens.size, MAX_SEQUENCE_LENGTH)",
        "        for (i in 0 until limit) {",
        "            ids[i] = vocab[tokens[i]] ?: 1",
        "        }",
        "        return ids",
        "    }",
        "}",
    ]
    with open(os.path.join(out_dir, "PriorityTokenizer.kt.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(kotlin_lines))


def save_vectorizer_examples(vectorizer, sample_texts, out_dir: str):
    vocab_lookup = make_vocab_lookup(vectorizer.get_vocabulary())
    encoded = encode_texts_with_vocab(sample_texts, vocab_lookup)
    payload = []
    for raw_text, token_ids in zip(sample_texts, encoded):
        payload.append({
            "raw_text": raw_text,
            "normalized_text": normalize_text_py(raw_text),
            "token_ids": token_ids.tolist(),
        })
    write_json(os.path.join(out_dir, "tokenizer_examples.json"), payload)


def convert_student_to_tflite(model: tf.keras.Model, out_dir: str) -> str:
    tflite_path = os.path.join(out_dir, "priority_mobile_regressor.tflite")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
    if ENABLE_DYNAMIC_RANGE_QUANT:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    tflite_model = converter.convert()
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    return tflite_path


def tflite_predict_token_ids(tflite_path: str, token_ids: np.ndarray) -> np.ndarray:
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_index = input_details[0]["index"]
    input_dtype = input_details[0]["dtype"]
    token_ids = np.asarray(token_ids, dtype=input_dtype)

    interpreter.resize_tensor_input(input_index, token_ids.shape, strict=False)
    interpreter.allocate_tensors()
    interpreter.set_tensor(input_index, token_ids)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]["index"])
    return np.clip(output.reshape(-1), 0.0, 1.0)


# =========================
# Main
# =========================
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Loading dataset...")
    df = load_dataframe(CSV_PATH)
    print(f"Loaded {len(df)} rows")

    train_df, val_df, test_df = split_dataframe(df)
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    train_labels = train_df[LABEL_COL].values.astype("float32")
    val_labels = val_df[LABEL_COL].values.astype("float32")
    test_labels = test_df[LABEL_COL].values.astype("float32")

    teacher_available = False
    teacher_metrics = None
    teacher_model = None
    teacher_train_pred = train_labels.copy()
    teacher_val_pred = val_labels.copy()
    teacher_test_pred = test_labels.copy()

    if TRAIN_TEACHER:
        print("Building teacher model...")
        try:
            teacher_model = build_teacher_model()
            teacher_model = compile_teacher_model(teacher_model)
            teacher_model.summary()

            teacher_callbacks = [
                tf.keras.callbacks.EarlyStopping(
                    monitor="val_mae",
                    patience=2,
                    mode="min",
                    restore_best_weights=True,
                ),
                tf.keras.callbacks.ModelCheckpoint(
                    filepath=os.path.join(OUTPUT_DIR, "best_teacher_model.keras"),
                    monitor="val_mae",
                    mode="min",
                    save_best_only=True,
                ),
            ]

            print("Training teacher...")
            teacher_history = teacher_model.fit(
                make_teacher_dataset(train_df, True),
                validation_data=make_teacher_dataset(val_df, False),
                epochs=TEACHER_MAX_EPOCHS,
                callbacks=teacher_callbacks,
            )

            teacher_train_pred = np.clip(
                teacher_model.predict(train_df["model_text"].tolist(), batch_size=TEACHER_BATCH_SIZE, verbose=0).reshape(-1),
                0.0,
                1.0,
            )
            teacher_val_pred = np.clip(
                teacher_model.predict(val_df["model_text"].tolist(), batch_size=TEACHER_BATCH_SIZE, verbose=0).reshape(-1),
                0.0,
                1.0,
            )
            teacher_test_pred = np.clip(
                teacher_model.predict(test_df["model_text"].tolist(), batch_size=TEACHER_BATCH_SIZE, verbose=0).reshape(-1),
                0.0,
                1.0,
            )

            teacher_metrics = regression_metrics(test_labels, teacher_test_pred)
            write_json(os.path.join(OUTPUT_DIR, "teacher_test_metrics.json"), teacher_metrics)
            write_json(os.path.join(OUTPUT_DIR, "teacher_history.json"), teacher_history.history)

            print("Teacher test metrics:")
            print(json.dumps(teacher_metrics, indent=2))
            teacher_available = True

        except Exception as e:
            print("Teacher stage failed. Falling back to pure mobile training without distillation.")
            print(repr(e))

    print("Building student tokenizer...")
    vectorizer = build_vectorizer(train_df["model_text"].tolist())
    actual_vocab_size = len(vectorizer.get_vocabulary())
    print(f"Actual vocabulary size: {actual_vocab_size}")

    verify_samples = [
        train_df["model_text"].iloc[0],
        val_df["model_text"].iloc[0],
        test_df["model_text"].iloc[0],
    ]
    verify_manual_tokenizer(vectorizer, verify_samples)
    print("Tokenizer verification passed. Android can reproduce the same token IDs using the exported vocab.")

    train_token_ids = vectorize_texts(vectorizer, train_df["model_text"].tolist())
    val_token_ids = vectorize_texts(vectorizer, val_df["model_text"].tolist())
    test_token_ids = vectorize_texts(vectorizer, test_df["model_text"].tolist())

    train_targets = train_labels.copy()
    if teacher_available:
        train_targets = np.clip(
            (1.0 - DISTILL_BLEND) * train_labels + DISTILL_BLEND * teacher_train_pred,
            0.0,
            1.0,
        ).astype("float32")
        print(f"Using blended student targets with distillation weight = {DISTILL_BLEND:.2f}")
    else:
        print("Teacher unavailable. Student will train directly on ground-truth labels.")

    student_model = build_student_model(actual_vocab_size)
    student_model = compile_student_model(student_model)
    student_model.summary()

    student_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_mae",
            patience=4,
            mode="min",
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_mae",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            mode="min",
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, "best_student_model.keras"),
            monitor="val_mae",
            mode="min",
            save_best_only=True,
        ),
    ]

    print("Training student...")
    student_history = student_model.fit(
        make_student_dataset(train_token_ids, train_targets, True),
        validation_data=make_student_dataset(val_token_ids, val_labels, False),
        epochs=STUDENT_MAX_EPOCHS,
        callbacks=student_callbacks,
    )

    print("Evaluating student...")
    student_test_pred = np.clip(
        student_model.predict(test_token_ids, batch_size=STUDENT_BATCH_SIZE, verbose=0).reshape(-1),
        0.0,
        1.0,
    )
    student_metrics = regression_metrics(test_labels, student_test_pred)
    print("Student test metrics:")
    print(json.dumps(student_metrics, indent=2))

    write_json(os.path.join(OUTPUT_DIR, "student_test_metrics.json"), student_metrics)
    write_json(os.path.join(OUTPUT_DIR, "student_history.json"), student_history.history)

    comparison_rows = test_df[[TITLE_COL, DESCRIPTION_COL, CATEGORY_COL, LABEL_COL]].copy()
    if teacher_available:
        comparison_rows["teacher_prediction"] = teacher_test_pred
    comparison_rows["student_prediction"] = student_test_pred
    comparison_rows.head(min(50, len(comparison_rows))).to_csv(
        os.path.join(OUTPUT_DIR, "sample_predictions.csv"),
        index=False,
    )

    save_vocabulary(vectorizer, OUTPUT_DIR)
    save_config(OUTPUT_DIR, actual_vocab_size, teacher_available)
    save_android_reference(OUTPUT_DIR)
    save_vectorizer_examples(vectorizer, verify_samples, OUTPUT_DIR)
    write_json(
        os.path.join(OUTPUT_DIR, "export_summary.json"),
        {
            "teacher_available": teacher_available,
            "teacher_metrics": teacher_metrics,
            "student_metrics": student_metrics,
            "actual_vocab_size": actual_vocab_size,
            "max_sequence_length": MAX_SEQUENCE_LENGTH,
            "artifacts": [
                "priority_mobile_regressor.tflite",
                "priority_mobile_regressor.keras",
                "vocabulary.txt",
                "training_config.json",
                "PriorityTokenizer.kt.txt",
                "tokenizer_examples.json",
                "sample_predictions.csv",
                "student_test_metrics.json",
                "student_history.json",
            ],
        },
    )

    student_model.save(os.path.join(OUTPUT_DIR, "priority_mobile_regressor.keras"))

    print("Converting student to TFLite...")
    tflite_path = convert_student_to_tflite(student_model, OUTPUT_DIR)
    print(f"TFLite written to: {tflite_path}")

    demo_token_ids = encode_texts_with_vocab(
        [
            "title: Large pothole category: Roads description: Deep pothole on a busy road damaging vehicles important_title: Large pothole",
            "title: Minor litter category: Waste description: Small amount of litter near a bench important_title: Minor litter",
        ],
        make_vocab_lookup(vectorizer.get_vocabulary()),
    )
    demo_scores = tflite_predict_token_ids(tflite_path, demo_token_ids)
    for idx, score in enumerate(demo_scores, start=1):
        print(f"Demo sample {idx} TFLite score: {float(score):.4f}")

    print("Done.")


if __name__ == "__main__":
    main()

Loading dataset...
Loaded 6000 rows
Train: 4200 | Val: 900 | Test: 900
Building model...
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text (InputLayer)           [(None,)]                    0         []                            
                                                                                                  
 preprocessing (KerasLayer)  {'input_type_ids': (None,    0         ['text[0][0]']                
                             128),                                                                
                              'input_mask': (None, 128)                                           
                             , 'input_word_ids': (None,                                           
                              128)}                                                               
     

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:

import shutil

shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)


'/content/priority_bert_outputs.zip'

In [ ]:

from google.colab import files

files.download(f"{OUTPUT_DIR}.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>